# Frozen AraBEST Mixture All on Kaggle — 4,000 vs. 8,000 clips

The 8K training split contains all 4K baseline rows, plus **200 new clips per dialect**. Both arms use frozen AraBEST-RQ-300M, the same mean pooling and Mixture All head, matched seeds, and 1,000 updates per seed. Each arm fits z-score statistics on its own training split. Both score on the same 200 ADI-20 dev rows and 6,819 Casablanca validation rows.

1. Import this notebook as private in Kaggle. Enable **Internet** and choose **GPU T4 ×2**. Feature extraction uses one GPU. CPU works if all required features are cached.
2. Attach **ADI20_4K_8K_Training_Manifests.zip** as a private dataset. The notebook also accepts the two extracted JSONL manifests. This ZIP already includes the 4K split.
3. Attach the saved **reusable_adi_cache** outputs and feature datasets used in earlier runs. With the 4K and evaluation caches attached, just the **4,000 new clips** need encoding. Check the cache counts before extraction.
4. To check your GPU budget first, set **PROBE_ONLY = True** and run the notebook through the extraction cell. It times up to 200 uncached training clips and prints a rough estimate for the remaining clips. To continue in the same session, set the flag to **False**, rerun the setup cell, then run from the extraction cell onward.
5. Download **adi20_4k_vs_8k_results.zip** from Output. The output includes the paired seed deltas in accuracy, macro-F1, and Jordanian metrics.

The previous 4K reference was **86.83% ADI-20 dev accuracy** and **57.99% Casablanca accuracy / 62.49% macro-F1**. The NADI blind test is outside this comparison.

Extraction defaults to a **6-hour cap per run**. If it pauses, save and attach the output containing `reusable_adi_cache` to resume. A network download already in progress can extend beyond the cap. Set `SPARSE_HF_READS = False` if sparse reads fail, or add the `HF_TOKEN` secret if authentication is required.

## 1. Set up Kaggle and load the pinned splits

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = Path("/tmp/adi-mixture-8k")
COMMIT = "b64437a37e7eda2f1721a56860278640bf085bdd"
REPO_URL = "https://github.com/alizbeeb5/Arabic-Dialect-Identification.git"

SEEDS = [490, 491, 492]
UPDATES = 1000                 # Same optimizer updates for both training sizes.
PROBE_ONLY = False             # True extracts up to 200 missing training clips and exits cleanly.
PROBE_CLIPS = 200
MAX_EXTRACT_HOURS = 6.0        # Per run, checked between shards and encode batches.
EXTRACT_BATCH = 4
EXTRACT_CHUNK = 48
SPARSE_HF_READS = True         # False downloads each needed ADI-20 shard in full.

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
WORK.mkdir(parents=True, exist_ok=True)

def git(*args):
    subprocess.run(["git", "-C", str(REPO), *args], check=True)

if not (REPO / ".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    git("init", "--quiet")
    git("remote", "add", "origin", REPO_URL)

pinned_commit = subprocess.run(
    ["git", "-C", str(REPO), "cat-file", "-e", f"{COMMIT}^{{commit}}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
if pinned_commit.returncode:
    git("fetch", "--quiet", "--depth=1", "origin", COMMIT)
git("checkout", "--quiet", "--detach", COMMIT)
sys.path.insert(0, str(REPO))

# Keep Kaggle's CUDA build of PyTorch and torchaudio.
packages = {"soundfile": "soundfile", "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub"}
missing_packages = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)

In [ ]:
import hashlib
import json
import shutil
import time
from zipfile import ZipFile
from collections import Counter, defaultdict
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score

from src.models import acoustic_encoder as ae
from src.experiments import acoustic_classifier as clf, layer_mixture as lm
from src.experiments import mfcc_svm_baseline as base
from src.experiments.casablanca_check import diagnostics

HEAD_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False

M = REPO / "data/manifests"
RUN = WORK / "scaling_4k_vs_8k_results" / time.strftime("%Y%m%d-%H%M%S")
CONFIG = json.loads((REPO / "configs/layer_mixture.json").read_text())
LABELS = json.loads((REPO / "configs/labels.json").read_text())["label_order"]
LABEL_ID = {label: i for i, label in enumerate(LABELS)}
JOR = LABEL_ID["JOR"]
OPT = CONFIG["optimization"]
ADI_REV = "c8da454c9f474bb075d3fe608940ee29e8ccfa43"
CAS_REV = "8951b1b88e28c1107142ced57967b8d16350951d"
MANIFESTS = {
    "train_scaling_4000": "ebeb36edce3399a3b3eb116ae3158185e1c9e6bc0308f270b3c45d9d6901f120",
    "train_scaling_8000": "c3e90a5441c1d70596c635c0a5e76147c88101b1498c22e76f167090fcea842d",
    "dev_pilot": "fb8684c35ef68e25834d5a4c82cddbd31044a899a63b665c41d3f9a179052d02",
    "casablanca_validation": "5ed7a36491ba6a41782302e1f3d31c3033b4d72ac18f1f7a2d04e85177dd064e",
}

def training_manifests():
    names = ("train_scaling_4000", "train_scaling_8000")
    decoded = {}
    for name in names:
        for path in sorted(INPUT.rglob(f"{name}.jsonl")):
            raw = path.read_bytes()
            if hashlib.sha256(raw).hexdigest() == MANIFESTS[name]:
                decoded[name] = raw
                break
    if len(decoded) < len(names):
        for package in sorted(INPUT.rglob("ADI20_4K_8K_Training_Manifests.zip")):
            with ZipFile(package) as archive:
                for name in names:
                    if name not in decoded:
                        raw = archive.read(f"{name}.jsonl")
                        if hashlib.sha256(raw).hexdigest() == MANIFESTS[name]:
                            decoded[name] = raw
            if len(decoded) == len(names):
                break
    if set(decoded) != set(names):
        raise ValueError("Attach and check ADI20_4K_8K_Training_Manifests.zip or both extracted manifests.")
    return decoded

training = training_manifests()

def read_rows(name):
    raw = training[name] if name in training else (M / f"{name}.jsonl").read_bytes()
    assert hashlib.sha256(raw).hexdigest() == MANIFESTS[name], f"Manifest mismatch: {name}"
    return [json.loads(line) for line in raw.splitlines() if line.strip()]

train4000 = read_rows("train_scaling_4000")
train8000 = read_rows("train_scaling_8000")
dev = read_rows("dev_pilot")
cas = read_rows("casablanca_validation")

for rows, per_label in ((train4000, 200), (train8000, 400)):
    assert Counter(row["label"] for row in rows) == {label: per_label for label in LABELS}
    assert all(row["split"] == "train" and row["source_revision"] == ADI_REV for row in rows)
    assert len({row["clip_id"] for row in rows}) == len(rows)

by_id = {row["clip_id"]: row for row in train8000}
base_ids = {row["clip_id"] for row in train4000}
assert all(by_id[row["clip_id"]] == row for row in train4000), "4K training rows changed in the 8K split"
added = [row for row in train8000 if row["clip_id"] not in base_ids]
assert Counter(row["label"] for row in added) == {label: 200 for label in LABELS}
for field in ("clip_id", "audio_sha256", "pcm_sha256"):
    training_values = {row[field] for row in train8000}
    eval_values = {row[field] for row in dev + cas if row.get(field)}
    assert len(training_values) == 8000
    assert training_values.isdisjoint(eval_values), f"Training/evaluation overlap by {field}"
assert len(dev) == 200 and len(cas) == 6819

print("Train: 4,000 and 8,000 nested clips. Eval: 200 ADI-20 dev and 6,819 Casablanca validation rows.")


## 2. Check cache coverage

In [ ]:
feature_config = clf.feature_config(CONFIG["encoder"])
provenance = lm.expected_provenance(feature_config)
write_cache = ae.FeatureCache(WORK / "reusable_adi_cache", provenance)
cas_ok = [row for row in cas if row.get("preparation_status", "ok") == "ok"]
assert len(cas_ok) == 6818

# Use attached caches in place. New parts go only into the writable cache.
read_caches = []
for config_path in sorted(INPUT.rglob("config.json")):
    index_path = config_path.with_name("index.jsonl")
    if config_path.parent.name != write_cache.root.name or not index_path.is_file():
        continue
    mounted = ae.FeatureCache(WORK, provenance)
    mounted.root, mounted.index_path = config_path.parent, index_path
    read_caches.append(mounted)

def feature_locations(rows):
    indices = [(cache.root, cache.index()) for cache in [*read_caches, write_cache]]
    locations = {}
    part_exists = {}
    for row in rows:
        clip_id = row["clip_id"]
        key = ae.clip_cache_key(clip_id, row["audio_sha256"], provenance)
        for root, index in indices:
            saved = index.get(clip_id)
            if saved is None or saved["cache_key"] != key:
                continue
            part = root / saved["part"]
            if part not in part_exists:
                part_exists[part] = part.is_file()
            if part_exists[part]:
                locations[clip_id] = (part, saved["row"], key)
                break
    return locations

def missing_features(rows):
    locations = feature_locations(rows)
    return [row for row in rows if row["clip_id"] not in locations]

def feature_batches(rows, batch_size=64):
    locations = feature_locations(rows)
    groups = defaultdict(list)
    for i, row in enumerate(rows):
        clip_id = row["clip_id"]
        part, position, key = locations[clip_id]
        groups[part].append((i, position, clip_id, key))
    for part, entries in groups.items():
        with np.load(part, allow_pickle=False) as data:
            values, ids, keys = data["features"], data["clip_ids"], data["cache_keys"]
            for start in range(0, len(entries), batch_size):
                batch = entries[start:start + batch_size]
                for _, position, clip_id, key in batch:
                    assert ids[position] == clip_id and keys[position] == key, f"Invalid cache row for {clip_id}"
                features = values[[item[1] for item in batch]].astype(np.float32, copy=False)
                assert np.isfinite(features).all(), f"Nonfinite features in {part}"
                yield [item[0] for item in batch], features

def load_features(rows):
    result = np.empty((len(rows), 26, 848), np.float32)
    for positions, values in feature_batches(rows):
        result[positions] = values
    return result

print("Attached feature caches:", len(read_caches))
for name, rows in (("ADI-20 train 4,000", train4000), ("New ADI-20 training clips", added), ("ADI-20 dev", dev), ("Casablanca", cas_ok)):
    missing = len(missing_features(rows))
    print(f"{name}: {len(rows) - missing}/{len(rows)} cached")


## 3. Encode only missing clips

In [ ]:
from importlib.metadata import PackageNotFoundError, version

encoder = None

def get_encoder():
    global encoder
    if encoder is None:
        try:
            installed = version("speechbrain")
        except PackageNotFoundError:
            installed = None
        if installed != "1.1.2":
            subprocess.run([
                sys.executable, "-m", "pip", "install", "-q", "--no-deps", "speechbrain==1.1.2",
            ], check=True)
        optional = {"hyperpyyaml": "hyperpyyaml", "sentencepiece": "sentencepiece", "joblib": "joblib"}
        needed = [package for module, package in optional.items() if importlib.util.find_spec(module) is None]
        if needed:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)
        encoder = ae.AcousticEncoder.from_pretrained(
            feature_config, device="cuda:0", cache_dir="/tmp/adi-hf-model"
        )
        clf.frozen_summary(encoder)
        assert json.loads(json.dumps(encoder.provenance())) == provenance, "Encoder provenance mismatch"
    return encoder

def optional_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return False

def encode_shard(handle, rows, scratch, label, deadline, on_encoded):
    import pyarrow.parquet as pq

    wanted = {int(row["source_row"]): row for row in rows}
    assert len(wanted) == len(rows), "Duplicate row in the pinned manifest"
    with pq.ParquetFile(handle) as parquet:
        offset = 0
        for group in range(parquet.num_row_groups):
            end = offset + parquet.metadata.row_group(group).num_rows
            selected = [i for i in wanted if offset <= i < end]
            if selected:
                position = offset
                for batch in parquet.iter_batches(
                    row_groups=[group], columns=["audio"], batch_size=EXTRACT_CHUNK
                ):
                    if time.monotonic() >= deadline:
                        return False
                    batch_end = position + batch.num_rows
                    block = []
                    for i in selected:
                        if position <= i < batch_end:
                            row = wanted.pop(i)
                            payload = batch.column(0).field("bytes")[i - position].as_py()
                            if not payload or hashlib.sha256(payload).hexdigest() != row["audio_sha256"]:
                                raise RuntimeError(f"Audio SHA256 mismatch for {row['clip_id']}")
                            path = ae.resolve_audio_path(scratch, row["audio_path"])
                            path.parent.mkdir(parents=True, exist_ok=True)
                            path.write_bytes(payload)
                            block.append(row)
                    if block:
                        _, failures = clf.prepare_features(
                            get_encoder(), write_cache, block, scratch,
                            {"batch_size": EXTRACT_BATCH, "chunk_size": EXTRACT_CHUNK}, label,
                        )
                        if failures:
                            with (WORK / "extraction_failures.jsonl").open("a", encoding="utf-8") as log:
                                for failure in failures:
                                    log.write(json.dumps(failure) + "\n")
                            raise RuntimeError(f"{len(failures)} failed features. See extraction_failures.jsonl.")
                        for row in block:
                            (scratch / row["audio_path"]).unlink()
                        on_encoded(len(block))
                    position = batch_end
                    if position > max(selected):
                        break
            offset = end
            if not wanted:
                break
    if wanted:
        raise RuntimeError(f"Pinned parquet did not contain {len(wanted)} requested rows.")
    return True

def fill_features(rows, label, deadline, limit=None):
    missing = sorted(
        missing_features(rows),
        key=lambda row: (row["source_dataset"], row["source_shard"], row["source_row"]),
    )
    if not missing:
        print(f"{label}: reusing all {len(rows)} cached clips")
        return True
    if not torch.cuda.is_available():
        raise RuntimeError("Select GPU T4 x2 or attach feature caches for the missing rows.")

    from huggingface_hub import HfFileSystem, hf_hub_download

    target = missing if limit is None else missing[:limit]
    buckets = defaultdict(list)
    for row in target:
        source, revision, shard = row["source_dataset"], row["source_revision"], row["source_shard"]
        if source == "UBC-NLP/Casablanca":
            assert revision == CAS_REV and "validation" in shard.lower() and "test" not in shard.lower()
        else:
            assert source == "UBC-NLP/NADI_2026_ADI20_micro" and revision == ADI_REV
        buckets[source, revision, shard].append(row)

    token = optional_hf_token()
    hub = HfFileSystem(token=token)
    started = time.monotonic()
    encoded = 0
    probe_size = min(PROBE_CLIPS, len(target))

    def record(count):
        nonlocal encoded
        previous = encoded
        encoded += count
        if previous < probe_size <= encoded:
            elapsed = time.monotonic() - started
            remaining_hours = elapsed * (len(missing) - encoded) / encoded / 3600
            print(
                f"{label}: {encoded} newly encoded in {elapsed / 60:.1f} min. "
                f"Rough estimate for remaining {len(missing) - encoded}: {remaining_hours:.2f} h.",
                flush=True,
            )

    for i, ((source, revision, shard), clips) in enumerate(buckets.items(), 1):
        if time.monotonic() >= deadline:
            return False
        print(f"{label}: shard {i}/{len(buckets)} | {len(clips)} clips | {shard}", flush=True)
        with TemporaryDirectory(prefix="adi-shard-") as directory:
            path = Path(directory)
            scratch = path / "audio"
            scratch.mkdir()
            if SPARSE_HF_READS and source == "UBC-NLP/NADI_2026_ADI20_micro":
                with hub.open(f"datasets/{source}@{revision}/{shard}", "rb", block_size=8 * 1024**2) as handle:
                    finished = encode_shard(handle, clips, scratch, label, deadline, record)
            else:
                local = hf_hub_download(
                    repo_id=source, repo_type="dataset", revision=revision, filename=shard,
                    local_dir=path / "parquet", token=token,
                )
                finished = encode_shard(local, clips, scratch, label, deadline, record)
            if not finished:
                return False
    assert not missing_features(target), f"{label}: feature cache is incomplete"
    return len(target) == len(missing)

In [ ]:
assert PROBE_CLIPS > 0 and MAX_EXTRACT_HOURS > 0
deadline = time.monotonic() + MAX_EXTRACT_HOURS * 3600
if PROBE_ONLY:
    fill_features(added, "New ADI-20 training timing probe", deadline, limit=PROBE_CLIPS)
    ready = False
    print("Probe finished. Set PROBE_ONLY = False to continue from the saved cache.")
else:
    ready = fill_features(train8000 + dev, "ADI-20 train and dev", deadline)
    if ready:
        ready = fill_features(cas_ok, "Casablanca validation", deadline)
    if not ready:
        print("Extraction budget reached. Save this run's output and attach it to resume in a new session.")

encoder = None
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Writable feature cache:", write_cache.root)

## 4. Train both heads with the same update budget

In [ ]:
def fit_head(features, labels, seed):
    init_seed, shuffle_seed = clf.run_seeds(seed)
    model = lm.build_model(tuple(features.shape), len(LABELS), init_seed).to(HEAD_DEVICE)
    optimizer = lm.make_optimizer(model, OPT)
    features, labels = features.to(HEAD_DEVICE), labels.to(HEAD_DEVICE)
    generator = torch.Generator().manual_seed(shuffle_seed)
    steps = 0
    while steps < UPDATES:
        order = torch.randperm(len(features), generator=generator)
        for indices in order.split(OPT["batch_size"]):
            indices = indices.to(HEAD_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(features[indices]), labels[indices])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), OPT["gradient_clip_norm"])
            optimizer.step()
            steps += 1
            if steps == UPDATES:
                break
    return model.cpu().eval()

TRAIN_ARMS = {"train_4000": train4000, "train_8000": train8000}
models = {}
normalization = {}
if ready:
    RUN.mkdir(parents=True, exist_ok=True)
    for arm, rows in TRAIN_ARMS.items():
        raw = load_features(rows)
        mean = raw.mean(axis=0)
        std = np.maximum(raw.std(axis=0), CONFIG["preprocessing"]["std_floor"])
        normalization[arm] = mean, std
        raw -= mean
        raw /= std
        features = torch.from_numpy(raw)
        labels = torch.tensor([LABEL_ID[row["label"]] for row in rows], dtype=torch.long)
        arm_dir = RUN / arm
        arm_dir.mkdir(exist_ok=True)
        np.savez(arm_dir / "zscore_stats.npz", mean=mean, std=std,
                 layer_names=np.array(provenance["layer_names"]))
        for seed in SEEDS:
            model = fit_head(features, labels, seed)
            models[arm, seed] = model
            torch.save(
                {"state_dict": model.state_dict(), "labels": LABELS, "layers": provenance["layer_names"],
                 "seed": seed, "updates": UPDATES, "train_size": len(rows), "config": CONFIG},
                arm_dir / f"head_{seed}.pt",
            )
            np.save(arm_dir / f"layer_weights_{seed}.npy", model.layer_weights().detach().numpy())
            print(f"{arm}, seed {seed}: trained for {UPDATES} updates", flush=True)
        del features, raw
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Evaluate both splits and export results

In [ ]:
@torch.no_grad()
def predict_split(readable, all_rows):
    valid_positions = [i for i, row in enumerate(all_rows) if row.get("preparation_status", "ok") == "ok"]
    assert len(valid_positions) == len(readable)
    predictions = {key: np.full(len(all_rows), -1, dtype=int) for key in models}
    for indices, batch in feature_batches(readable, batch_size=128):
        output_positions = [valid_positions[i] for i in indices]
        for arm, (mean, std) in normalization.items():
            features = torch.from_numpy((batch - mean) / std)
            for seed in SEEDS:
                predictions[arm, seed][output_positions] = models[arm, seed](features).argmax(1).numpy()
    for values in predictions.values():
        assert (values[valid_positions] >= 0).all(), "Missing prediction on readable audio"
    return predictions

def score(rows, pred):
    gold = np.array([LABEL_ID[row["label"]] for row in rows])
    present = np.unique(gold)
    jordanian = gold == JOR
    predicted_jor = pred == JOR
    correct = gold == pred
    names = [LABELS[i] if i >= 0 else "FAILURE" for i in pred]
    region = diagnostics(rows, [name if name != "FAILURE" else None for name in names],
                         [LABELS[i] for i in present], LABELS)[0]["region_accuracy"]
    metrics = {
        "accuracy": correct.mean(),
        "macro_f1": f1_score(gold, pred, labels=present, average="macro", zero_division=0),
        "jor_recall": correct[jordanian].mean(),
        "jor_precision": correct[predicted_jor].mean() if predicted_jor.any() else 0.0,
        "non_jor_accuracy": correct[~jordanian].mean(),
        "region_accuracy": region,
        "n": len(gold),
        "failures": int((pred < 0).sum()),
    }
    return gold, names, metrics

if ready:
    records = []
    for split, readable, rows in (("ADI20_dev", dev, dev), ("Casablanca", cas_ok, cas)):
        predictions = predict_split(readable, rows)
        for (arm, seed), pred in predictions.items():
            gold, names, metrics = score(rows, pred)
            records.append({"arm": arm, "seed": seed, "split": split,
                            "train_size": len(TRAIN_ARMS[arm]), **metrics})
            folder = RUN / arm / str(seed)
            folder.mkdir(exist_ok=True)
            pd.DataFrame({"clip_id": [row["clip_id"] for row in rows],
                          "gold": [LABELS[i] for i in gold], "prediction": names}).to_csv(
                folder / f"{split}_predictions.csv", index=False,
            )
            labels = LABELS + ["FAILURE"]
            cm = confusion_matrix(gold, np.where(pred >= 0, pred, len(LABELS)), labels=range(len(labels)))
            pd.DataFrame(cm, index=labels, columns=labels).to_csv(folder / f"{split}_confusion.csv")

    results = pd.DataFrame(records)
    metrics = ["accuracy", "macro_f1", "jor_recall", "jor_precision", "non_jor_accuracy", "region_accuracy"]
    summary = results.groupby(["split", "arm"])[metrics].agg(["mean", "std"])
    baseline = results[results.arm == "train_4000"].set_index(["split", "seed"])
    scaled = results[results.arm == "train_8000"].set_index(["split", "seed"])
    deltas = 100 * (scaled[metrics] - baseline[metrics])
    results.to_csv(RUN / "results.csv", index=False)
    summary.to_csv(RUN / "summary.csv")
    deltas.to_csv(RUN / "paired_deltas_percentage_points.csv")
    display((100 * summary).round(2))
    print("8,000 minus 4,000 clips, mean percentage points across matched seeds")
    display(deltas.groupby(level="split").mean().round(2))

    protocol = {
        "repo_commit": COMMIT, "encoder": CONFIG["encoder"], "manifest_sha256": MANIFESTS,
        "seeds": SEEDS, "optimizer_updates_per_head": UPDATES, "head_device": HEAD_DEVICE,
        "normalization": "Fit independently on the training clips of each arm.",
        "previous_4k_reference": {"adi20_dev_accuracy": 0.8683, "casablanca_accuracy": 0.5799, "casablanca_macro_f1": 0.6249},
        "evaluation": "Same 200 ADI-20 dev rows and 6,819 Casablanca validation rows for both arms.",
        "speaker_independence_verified": False,
    }
    (RUN / "protocol.json").write_text(json.dumps(protocol, indent=2))
    archive = shutil.make_archive(str(WORK / "adi20_4k_vs_8k_results"), "zip", RUN)
    print("Download from Kaggle Output:", archive)

## 6. Test on SHAMS LangMedia — no retraining

This runs the saved 4K and 8K heads on the 907 LangMedia speech clips in [SHAMS](https://huggingface.co/datasets/shams-nlp/shams). It uses 168 Jordanian, 215 Syrian, 316 Lebanese and 208 Palestinian clips. Predictions retain all 20 ADI-20 classes. The reported macro-F1 averages over the four reference dialects.

Run the cell after the main run, or attach `adi20_4k_vs_8k_results.zip` in a fresh Kaggle session and run only the first setup code cell and this block. Enable Internet and use a GPU to encode the 907 new clips. The block saves features for later reuse and exports per-seed results, confusion matrices, and class scores to `shams_langmedia_results.zip`. Cross-dataset source and speaker independence is unverified.


In [ ]:
# Run after the 4K-vs-8K notebook's first setup cell. Attach the saved results ZIP.
from pathlib import Path
import importlib.util
import io
import json
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from zipfile import ZipFile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score

SHAMS_RESULTS_ZIP = None  # Optional absolute path if more than one matching archive is attached.
SHAMS_DATA = "shams-nlp/shams"
SHAMS_REV = "0d4aa7dba621f5483c372eac718d4ecce5b1b4e3"
SHAMS_COUNTRIES = {"jordan": "JOR", "lebanon": "LEB", "palestine": "PAL", "syria": "SYR"}
SHAMS_SUPPORT = {"JOR": 168, "LEB": 316, "PAL": 208, "SYR": 215}


def shams_checkpoint_archive(working, inputs, explicit=None):
    if explicit is not None:
        path = Path(explicit)
        if path.is_file():
            return path
        raise FileNotFoundError(path)
    name = "adi20_4k_vs_8k_results.zip"
    path = working / name
    if path.is_file():
        return path
    candidates = sorted(p for p in inputs.rglob(name) if p.is_file())
    if len(candidates) == 1:
        return candidates[0]
    raise FileNotFoundError(f"Attach {name} or set SHAMS_RESULTS_ZIP to its exact path.")


def shams_extract_features(path, feature_config, device, encoder_module):
    try:
        installed = version("speechbrain")
    except PackageNotFoundError:
        installed = None
    if installed != "1.1.2":
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "speechbrain==1.1.2"],
                       check=True)
    deps = [name for name in ("hyperpyyaml", "sentencepiece", "joblib") if importlib.util.find_spec(name) is None]
    if deps:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)

    encoder = encoder_module.AcousticEncoder.from_pretrained(feature_config, device=device, cache_dir="/tmp/adi-hf-model")
    data = hf_hub_download(SHAMS_DATA, repo_type="dataset", revision=SHAMS_REV,
                           filename="data/test-00000-of-00001.parquet")
    labels, clip_ids, groups, blocks = [], [], [], []
    columns = ["audio", "seg_id", "source", "country", "original_file"]

    with pq.ParquetFile(data) as parquet:
        for batch in parquet.iter_batches(batch_size=48, columns=columns):
            clips = []
            for row in batch.to_pylist():
                if row["source"].strip().lower() != "langmedia":
                    continue
                clip_id = str(row["seg_id"])
                country = SHAMS_COUNTRIES[row["country"].strip().lower()]
                payload = (row["audio"] or {}).get("bytes")
                if not payload:
                    raise ValueError(f"Missing LangMedia audio for {clip_id}")
                wave, sr = sf.read(io.BytesIO(payload), dtype="float32", always_2d=True)
                prepared = encoder_module.prepare_waveform(wave.T, sr, feature_config)
                clips.append((clip_id, prepared))
                clip_ids.append(clip_id)
                labels.append(country)
                groups.append(row["original_file"] or "")
            if clips:
                features = encoder.extract(clips, batch_size=4 if device.startswith("cuda") else 1)
                assert features.clip_ids == [clip_id for clip_id, _ in clips]
                blocks.append(features.features)
            if len(blocks) and len(blocks) % 5 == 0 and clips:
                print(f"Encoded {len(clip_ids)}/907 LangMedia clips", flush=True)

    raw = np.concatenate(blocks)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez(tmp, features=raw, clip_ids=np.array(clip_ids), labels=np.array(labels),
             recordings=np.array(groups), revision=np.array(SHAMS_REV),
             layer_names=np.array(encoder.layer_names))
    tmp.replace(path)


def shams_predictions(raw, mean, std, model):
    with torch.inference_mode():
        return np.concatenate([
            model(torch.from_numpy(np.asarray((batch - mean) / std, dtype=np.float32))).numpy()
            for batch in np.array_split(raw, (len(raw) + 127) // 128)
        ])


def shams_metrics(gold, pred):
    jordanian = gold == "JOR"
    predicted_jor = pred == "JOR"
    return {
        "accuracy": (pred == gold).mean(),
        "macro_f1": f1_score(gold, pred, labels=list(SHAMS_SUPPORT), average="macro", zero_division=0),
        "jor_recall": (pred[jordanian] == "JOR").mean(),
        "jor_precision": (gold[predicted_jor] == "JOR").sum() / max(predicted_jor.sum(), 1),
        "jor_to_syr_count": int((pred[jordanian] == "SYR").sum()),
        "n": len(gold),
    }


def run_shams_test():
    from src.experiments import acoustic_classifier as clf, layer_mixture as lm
    from src.models import acoustic_encoder as ae

    archive_path = shams_checkpoint_archive(WORK, INPUT, SHAMS_RESULTS_ZIP)
    labels = json.loads((REPO / "configs/labels.json").read_text())["label_order"]
    results_dir = WORK / "shams_langmedia_results"
    results_dir.mkdir(exist_ok=True)

    with ZipFile(archive_path) as archive:
        protocol = json.loads(archive.read("protocol.json"))
        if protocol["repo_commit"] != COMMIT:
            raise ValueError("The saved heads were built at a different code revision.")
        feature_config = clf.feature_config(protocol["encoder"])
        cache_name = f"shams_{SHAMS_REV[:12]}_{feature_config.sha256()[:16]}.npz"
        cache = WORK / cache_name
        if not cache.is_file():
            cache = next(INPUT.rglob(cache_name), cache)
        if not cache.is_file():
            device = "cuda:0" if torch.cuda.is_available() else "cpu"
            shams_extract_features(cache, feature_config, device, ae)
        with np.load(cache, allow_pickle=False) as saved:
            raw, gold = saved["features"], saved["labels"]
            clip_ids, recordings = saved["clip_ids"], saved["recordings"]
            layer_names = saved["layer_names"].tolist()
            assert saved["revision"].item() == SHAMS_REV

        assert pd.Series(gold).value_counts().to_dict() == SHAMS_SUPPORT
        assert len(set(clip_ids)) == len(clip_ids) == len(gold) == len(raw) == 907
        assert raw.dtype == np.float32 and raw.shape == (907, len(layer_names), 848)
        assert np.isfinite(raw).all()

        summary, per_dialect = [], []
        for arm in ("train_4000", "train_8000"):
            with np.load(io.BytesIO(archive.read(f"{arm}/zscore_stats.npz")), allow_pickle=False) as stats:
                mean, std = stats["mean"], stats["std"]
                assert stats["layer_names"].tolist() == layer_names
            assert mean.shape == std.shape == raw.shape[1:] and np.isfinite(mean).all()
            assert np.isfinite(std).all() and (std > 0).all()
            for seed in protocol["seeds"]:
                payload = archive.read(f"{arm}/head_{seed}.pt")
                head = torch.load(io.BytesIO(payload), map_location="cpu", weights_only=True)
                assert head["labels"] == labels and head["layers"] == layer_names
                assert head["seed"] == seed and head["train_size"] == int(arm.split("_")[1])
                assert clf.feature_config(head["config"]["encoder"]) == feature_config
                net = lm.build_model(tuple(raw.shape), len(labels), 0)
                net.load_state_dict(head["state_dict"])
                logits = shams_predictions(raw, mean, std, net.eval())
                pred = np.asarray(labels)[logits.argmax(axis=1)]

                sample = pd.DataFrame(dict(clip_id=clip_ids, recording=recordings, gold=gold, prediction=pred))
                sample.join(pd.DataFrame(logits, columns=[f"logit_{label}" for label in labels])).to_csv(
                    results_dir / f"{arm}_seed_{seed}_predictions.csv", index=False
                )
                conf = confusion_matrix(gold, pred, labels=labels)
                pd.DataFrame(conf, index=labels, columns=labels).loc[list(SHAMS_SUPPORT)].to_csv(
                    results_dir / f"{arm}_seed_{seed}_confusion.csv"
                )
                summary.append({"arm": arm, "seed": seed, **shams_metrics(gold, pred)})
                for label, n in SHAMS_SUPPORT.items():
                    per_dialect.append(dict(arm=arm, seed=seed, label=label, n=n,
                                            recall=(pred[gold == label] == label).mean()))

    stats = pd.DataFrame(summary)
    stats.to_csv(results_dir / "results.csv", index=False)
    pd.DataFrame(per_dialect).to_csv(results_dir / "per_dialect.csv", index=False)
    (results_dir / "protocol.json").write_text(json.dumps({
        "dataset": SHAMS_DATA, "revision": SHAMS_REV, "subset": "LangMedia only",
        "support": SHAMS_SUPPORT, "model_outputs": labels, "training_protocol": protocol,
        "source_and_speaker_overlap_with_adi20_training_verified": False,
    }, indent=2))
    display(stats.groupby("arm")[["accuracy", "macro_f1", "jor_recall", "jor_precision"]].mean().mul(100).round(2))
    display(stats.pivot(index="seed", columns="arm", values="jor_to_syr_count"))
    bundle = shutil.make_archive(str(WORK / "shams_langmedia_results"), "zip", results_dir)
    print("Download from Kaggle Output:", bundle)


if not all(key in globals() for key in ("WORK", "INPUT", "REPO", "COMMIT")):
    raise RuntimeError("Run the 4K-vs-8K notebook's first setup code cell, then run this block.")
run_shams_test()
